# 41 · The boundary repair, against shapely

**Question.** Fieldwork experiment 194 made the study area remove a vertex equal to the one before it, and the tip of
a zero-width spike, from a published boundary as it is read, and list each removal. On OCHA COD-AB's 16 districts of
Sierra Leone and geoBoundaries' 92 municipalities of Rio de Janeiro state, does shapely drop the same repeated
vertices, does the spike rule written out here remove the same tips, and does no area change?

Removals are compared exactly, position by position and in order; an area is held to **1e-12** square degrees.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["shapely"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

The repeats are shapely's `remove_repeated_points`. The spike rule has no shapely equivalent, so it is written out
from the fixture's declared order and on-segment test.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_41_boundary_repair import run

report = run(Path.cwd() / "fixtures" / "boundary-repair-001.json")
print("agrees:", report["agrees"])
for level in report["levels"]:
    print(f"\n{'OK ' if level['agrees'] else 'DIFF'} {level['id']}: {level['units']} units, {level['repaired']} repaired, "
          f"{level['repeated']} repeated vertices, {level['spikes']} spike tips")
    for u in level["repairedUnits"]:
        print(f"  {u['name']}: {u['shapely']['repeated']} repeated, {len(u['shapely']['spike'])} spike; "
              f"{u['vertices']['published']} -> {u['vertices']['repaired']} vertices; area change {u['areaChange']:.1e}")
    for d in level["differs"]:
        print("  DIFF", d["name"], d["checks"])

## What GEOS says

GEOS treats a repeated vertex as valid and a spike as a self-intersection, so before the repair it refuses one unit,
Bonthe, and Fieldwork's own check refused fourteen. Recorded, not asserted.

In [ ]:
for level in report["levels"]:
    print(level["id"], "· GEOS invalid before:", level["geosInvalidBefore"], "· after:", level["geosInvalidAfter"] or "none")
    print("  decided differently by exact arithmetic:", level["inexact"] or "nowhere")

## What this cannot establish

**That the published boundaries have no other fault**: a ring crossing itself is not repaired. **That a spike is a
digitising error**: it encloses no area, which is why it can be removed. **How often the repair runs on geoBoundaries
in general**: one level of one country is here.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("shapely :", report["environment"]["shapely"], "· GEOS", report["environment"]["geos"])
print("runtime :", report["environment"]["runtime"])